<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebboks/tabformer_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Checking the TabFormer native-column result for leakage

In `cross_dataset_protocol.ipynb`, all 18 native columns of TabFormer score a test PR-AUC of 0.887, far above our six features with location (0.346). Before we report this result, we check whether part of it comes from leakage rather than from behaviour the model could use on new data. We check three things:

1. **The `Errors?` column.** It records problems with a transaction, such as a bad PIN or insufficient balance. If some error values are only recorded for fraud, the column gives the answer away.
2. **Merchant memorisation.** If the model learns which merchants had fraud in the training period, it can score well on the same merchants later without learning behaviour. We check which merchant fields the native run uses, how often test fraud happens at merchants that already had fraud in training, and how much of the result depends on the merchant fields.
3. **The same users in training and test.** Our split is by time, so every test user also has transactions in the training period. A model can then learn something about each user rather than about fraud. We retrain with each test user's transactions left out of training.

**Runs** (all with the same data, features, split and settings as `cross_dataset_protocol.ipynb`: Random Forest Tuned, five seeds, F2 threshold on validation, 500 day-block bootstrap resamples):

| Run | What changes |
|---|---|
| All native columns | nothing; loaded from `cross_dataset_protocol.ipynb` |
| Without `Errors?` | `Errors?` is removed |
| Without merchant fields | `Merchant State` is removed (`Merchant Name`, `Merchant City` and `Zip` are never used) |
| Unseen users, all native columns | each test user is scored by a model trained without that user |
| Unseen users, six + location | the same, for our six features with location |

All runs are scored on the same resampled test days, so each difference from the full native run has a 95% interval.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook. We use the same `QUICK` setting as in `cross_dataset_protocol.ipynb`, so the saved native run is found.

In [1]:
# 1.1 Settings
DATASETS = ['tabformer']
QUICK    = False                                 # True = 1 seed, 100 trees, 200 bootstrap resamples (a fast preview only)

SEEDS     = [42] if QUICK else [42, 43, 44, 45, 46]
N_TREES   = 100 if QUICK else 200
N_BOOT    = 200 if QUICK else 500
BOOT_SEED = 2026

USER_FOLDS = 5         # every user is left out of training once
FOLD_SEED = 2026

# TabFormer: the same user sample and window as tabformer_validation.ipynb
TF_N_USERS, TF_SAMPLE_SEED, TF_WINDOW_START = 400, 42, '2015-01-01'

KAGGLE_SLUGS = {
    'sparkov':   'kartik2112/fraud-detection',
    'tabformer': 'ealtman2019/credit-card-transactions',
    'cc2026':    'mohankrishnathalla/credit-card-fraud-detection-dataset-2026',
}
NAMES = {'sparkov': 'Sparkov', 'tabformer': 'TabFormer', 'cc2026': 'CC Fraud 2026'}
assert all(ds in KAGGLE_SLUGS for ds in DATASETS)
print(f'datasets: {DATASETS} | seeds: {SEEDS} | trees: {N_TREES} | bootstrap: {N_BOOT}')

datasets: ['tabformer'] | seeds: [42, 43, 44, 45, 46] | trees: 200 | bootstrap: 500


In [2]:
# 1.2 Download the datasets from Kaggle
import os, glob
from getpass import getpass

def data_dir(ds):
    return f'/content/data/{ds}'

for ds in DATASETS:
    if not glob.glob(f'{data_dir(ds)}/**/*.csv', recursive=True):
        if 'KAGGLE_KEY' not in os.environ:
            os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
            os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')
        !kaggle datasets download -d {KAGGLE_SLUGS[ds]} -p {data_dir(ds)} --unzip -q
    for f in sorted(glob.glob(f'{data_dir(ds)}/**/*', recursive=True)):
        if os.path.isfile(f):
            print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions
License(s): other
      1.9 MB  /content/data/tabformer/User0_credit_card_transactions.csv
   2350.7 MB  /content/data/tabformer/credit_card_transactions-ibm_v2.csv
      0.5 MB  /content/data/tabformer/sd254_cards.csv
      0.2 MB  /content/data/tabformer/sd254_users.csv


In [3]:
# 1.3 Clone the project repository to use our helpers
REPO = 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.exists('src/evaluation.py'):
    if not os.path.exists(REPO):
        github_token = getpass('Paste your GitHub token: ').strip()
        !git clone https://{github_token}@github.com/subbareddypullalacheruvu/{REPO}.git
    %cd {REPO}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 381 (delta 156), reused 53 (delta 44), pack-reused 149 (from 1)
Receiving objects: 100% (381/381), 11.99 MiB | 22.04 MiB/s, done.
Resolving deltas: 100% (234/234), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Imports and the Google Drive folders for progress and results
import sys, gc
import numpy as np
from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk, haversine_distance
from src.evaluation import chronological_val_split, run_seeds, bootstrap_model, summarise, paired_difference, point_metrics, format_ci
pd.set_option('display.max_columns', 60)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol'
RESULTS_DIR = f'{BASE}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def checkpoint_dir(ds):
    path = f"{BASE}/{ds}{'_quick' if QUICK else ''}"
    os.makedirs(path, exist_ok=True)
    return path

for ds in DATASETS:
    print(f'progress ({ds}) ->', checkpoint_dir(ds))
print('results ->', RESULTS_DIR)

Mounted at /content/drive
progress (tabformer) -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/tabformer
results -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results


## 2. Loading the data, building the features and splitting by time
These cells are the same as Sections 2 to 6 of `cross_dataset_protocol.ipynb`, so the test rows are in the same order as its saved predictions.

In [5]:
# 2.1 One loader per dataset
def csv_files(ds):
    return glob.glob(f'{data_dir(ds)}/**/*.csv', recursive=True)

def load_sparkov():
    files = csv_files('sparkov')
    tr = pd.read_csv(next(f for f in files if f.endswith('fraudTrain.csv')))
    te = pd.read_csv(next(f for f in files if f.endswith('fraudTest.csv')))
    tr['_split'], te['_split'] = 'train', 'test'          # Kaggle's own train/test files, as in all our notebooks
    d = pd.concat([tr, te], ignore_index=True)
    d = d.drop(columns=[c for c in d.columns if c.startswith('Unnamed')])
    d['_time'] = pd.to_datetime(d['trans_date_trans_time'])
    d['_y'] = d['is_fraud'].astype(int)
    d['_card'] = d['cc_num'].astype(str)
    d['_amount'] = d['amt'].astype(float)
    d['_category'] = d['category'].astype(str)
    d['age'] = (d['_time'] - pd.to_datetime(d['dob'])).dt.days / 365.25
    d['gender_flag'] = (d['gender'] == 'M').astype(int)
    skip = {'trans_date_trans_time', 'cc_num', 'first', 'last', 'street', 'trans_num', 'unix_time',
            'dob', 'is_fraud', '_split'}
    native = [c for c in tr.columns if c not in skip and not c.startswith('Unnamed')]
    text = {'category', 'gender', 'city', 'state', 'job', 'merchant', 'zip'}
    return d, native, text

def load_tabformer():
    files = csv_files('tabformer')
    users_csv = next(f for f in files if 'user' in os.path.basename(f).lower()
                     and 'transaction' not in os.path.basename(f).lower())
    txn_csv = max((f for f in files if 'transaction' in os.path.basename(f).lower()), key=os.path.getsize)
    users = pd.read_csv(users_csv)
    users['User'] = np.arange(len(users))               # transactions refer to users by row number
    rng = np.random.default_rng(TF_SAMPLE_SEED)
    sampled = set(rng.choice(users['User'].values, size=min(TF_N_USERS, len(users)), replace=False).tolist())
    parts = [ch[ch['User'].isin(sampled)] for ch in pd.read_csv(txn_csv, chunksize=2_000_000, low_memory=False)]
    raw = pd.concat(parts, ignore_index=True)
    txn_cols = list(raw.columns)

    d = raw.copy()
    d['_time'] = pd.to_datetime(dict(year=d['Year'], month=d['Month'], day=d['Day']), errors='coerce') \
                 + pd.to_timedelta(d['Time'].astype(str) + ':00', errors='coerce')
    d['_y'] = d['Is Fraud?'].astype(str).str.strip().str.lower().isin(['yes', '1', 'true']).astype(int)
    d['_amount'] = pd.to_numeric(d['Amount'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    d['_card'] = d['User'].astype(str) + '_' + d['Card'].astype(str)     # one card = user + card number
    d['_category'] = d['MCC'].astype(str)
    user_cols = [c for c in users.columns if c not in d.columns]
    d = d.merge(users[['User'] + user_cols], on='User', how='left')
    birth = pd.to_datetime(dict(year=d['Birth Year'], month=d['Birth Month'], day=1), errors='coerce')
    d['age'] = (d['_time'] - birth).dt.days / 365.25
    d['gender_flag'] = d['Gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])

    skip = {'User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Is Fraud?', 'Merchant Name', 'Zip',
            'Person', 'Address', 'Apartment', 'Zipcode', 'Birth Month'}
    native = [c for c in txn_cols + user_cols if c not in skip]
    text = {'MCC', 'Merchant State', 'Merchant City', 'State', 'Gender', 'City', 'Use Chip', 'Errors?'}
    return d, native, text

def load_cc2026():
    d = pd.read_csv(max(csv_files('cc2026'), key=os.path.getsize), low_memory=False)
    raw_cols = list(d.columns)
    d['_time'] = pd.to_datetime(d['transaction_datetime'], errors='coerce')
    d['_y'] = pd.to_numeric(d['is_fraud'], errors='coerce').fillna(0).astype(int)
    d['_card'] = d['card_id'].astype(str)
    d['_amount'] = d['transaction_amount'].astype(float)
    d['_category'] = d['merchant_category'].astype(str)
    d['age'] = pd.to_numeric(d['customer_age'], errors='coerce')
    d['gender_flag'] = d['customer_gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])
    # identifiers, the time stamp, the label and the two columns that describe the fraud itself
    skip = {'transaction_id', 'customer_id', 'card_id', 'merchant_id', 'transaction_datetime',
            'is_fraud', 'fraud_type', 'fraud_amount_lost'}
    native = [c for c in raw_cols if c not in skip]
    text = {'merchant_category', 'merchant_country', 'transaction_day_of_week', 'customer_gender',
            'customer_state', 'transaction_channel', 'device_type', 'browser', 'email_domain'}
    return d, native, text

LOADERS = {'sparkov': load_sparkov, 'tabformer': load_tabformer, 'cc2026': load_cc2026}

In [6]:
# 3.1 Time gap, distance, velocity and location features
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]

def build_features(d, ds):
    d = d.sort_values(['_card', '_time']).reset_index(drop=True)
    by_card = d.groupby('_card')
    d['_gap_h'] = by_card['_time'].diff().dt.total_seconds() / 3600

    if ds == 'sparkov':
        prev_lat, prev_lon = by_card['merch_lat'].shift(1), by_card['merch_long'].shift(1)
        d['_dist'] = haversine_distance(d['merch_lat'], d['merch_long'], prev_lat, prev_lon)
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)          # +1 second so we never divide by zero
        d['dist_home_log'] = np.log1p(haversine_distance(d['lat'], d['long'], d['merch_lat'], d['merch_long']))
        LOCATION = ['dist_home_log', 'dist_log']

    elif ds == 'tabformer':
        # no GPS points: distance proxy = 1 if the merchant city changed since the card's previous transaction
        city = d['Merchant City'].astype(str)
        prev_city = by_card['Merchant City'].shift(1)
        d['_dist'] = np.where(prev_city.isna(), np.nan, (city != prev_city.astype(str)).astype(float))
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 60)            # +1 minute (TabFormer's clock resolution)
        ms, has = d['Merchant State'], d['Merchant State'].notna()
        d['abroad'] = (has & (ms.astype(str).str.len() > 2)).astype(int)          # foreign merchants have a country name
        d['outside_home_state'] = (has & (ms.astype(str) != d['State'].astype(str))).astype(int)
        LOCATION = ['outside_home_state', 'abroad']
        d = d[d['_time'] >= TF_WINDOW_START].copy()                # gaps were computed on the full history first

    elif ds == 'cc2026':
        d['_dist'] = pd.to_numeric(d['distance_from_last_tx_km'], errors='coerce')
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)
        d['dist_home_log'] = np.log1p(pd.to_numeric(d['distance_from_home_km'], errors='coerce').clip(lower=0))
        LOCATION = ['dist_home_log', 'dist_log', 'international_transaction', 'high_risk_country', 'merchant_country_risk']

    d['gap_log'] = np.log1p(d['_gap_h']).fillna(-1)                # -1 = the card's first transaction
    d['dist_log'] = np.log1p(d['_dist'].clip(lower=0)).fillna(-1)
    d['velocity_log'] = np.log1p(d['_vel'].clip(lower=0)).fillna(-1)
    d['amt_log'] = np.log1p(d['_amount'].clip(lower=0)).fillna(0)
    d['is_night'] = d['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
    d['age'] = d['age'].fillna(-1)
    d['day'] = d['_time'].dt.floor('D')
    d = d.sort_values('_time').reset_index(drop=True)
    return d, LOCATION

def gap_check(d, ds):
    check = pd.DataFrame({'fraud': d.loc[d['_y'] == 1, ['_gap_h', '_dist']].median(),
                          'normal': d.loc[d['_y'] == 0, ['_gap_h', '_dist']].median()})
    check.index = ['median time gap (hours)', 'median distance' + (' (city changed share)' if ds == 'tabformer' else ' (km)')]
    return check

In [7]:
# 4.1 Train / validation / test
def split_by_time(d, ds):
    if ds == 'sparkov':
        train, test = d[d['_split'] == 'train'], d[d['_split'] == 'test'].copy()
    else:
        cutoff_test = d['_time'].min() + 0.8 * (d['_time'].max() - d['_time'].min())
        train, test = d[d['_time'] < cutoff_test], d[d['_time'] >= cutoff_test].copy()
    train_sub, val, _ = chronological_val_split(train, '_time', val_frac=0.2)
    for name, part in [('train_sub', train_sub), ('val', val), ('test', test)]:
        print(f"{name:9s} {part['_time'].min()} to {part['_time'].max()}  rows={len(part):>9,}  "
              f"fraud={part['_y'].sum():>6,}  rate={part['_y'].mean():.4%}")
    return train_sub, val, test

In [8]:
# 5.1 category_risk and the native-column encodings
def as_number(s):
    return pd.to_numeric(s.astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')

def encode(train_sub, val, test, ds, NATIVE_COLS, TEXT_COLS):
    parts = (train_sub, val, test)
    risk_map = fit_category_risk(train_sub, '_category', '_y')
    for p in parts:
        p['category_risk'] = apply_category_risk(p, '_category', risk_map)

    if ds == 'cc2026':
        country_map = fit_category_risk(train_sub, 'merchant_country', '_y')
        for p in parts:
            p['merchant_country_risk'] = apply_category_risk(p, 'merchant_country', country_map)

    NATIVE_FEATURES, native_info = [], []
    y_fit, prior = train_sub['_y'], train_sub['_y'].mean()
    for col in NATIVE_COLS:
        name, s = f'raw__{col}', train_sub[col]
        if col not in TEXT_COLS and pd.api.types.is_bool_dtype(s):
            for p in parts: p[name] = p[col].astype(float)
            used = 'numeric'
        elif col not in TEXT_COLS and pd.api.types.is_numeric_dtype(s):
            fill = s.median()
            for p in parts: p[name] = p[col].astype(float).fillna(fill)
            used = 'numeric'
        elif col not in TEXT_COLS and as_number(s).notna().mean() > 0.95:       # e.g. '$1,234'
            fill = as_number(s).median()
            for p in parts: p[name] = as_number(p[col]).fillna(fill)
            used = 'numeric (parsed text)'
        elif s.astype(str).nunique() <= 1000:
            rates = y_fit.groupby(s.astype(str)).mean()
            for p in parts: p[name] = p[col].astype(str).map(rates).fillna(prior).astype(float)
            used = 'fraud rate of each value'
        else:
            native_info.append({'column': col, 'used as': 'left out (more than 1,000 values)'})
            continue
        NATIVE_FEATURES.append(name)
        native_info.append({'column': col, 'used as': used})
    print(f'all-native run uses {len(NATIVE_FEATURES)} columns')
    return NATIVE_FEATURES, pd.DataFrame(native_info)

In [9]:
# 6.1 Feature groups, training and the day-block bootstrap
SIX = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']

def feature_groups(LOCATION, NATIVE_FEATURES):
    return {
        'Distance only':      ['dist_log'],
        'Time gap only':      ['gap_log'],
        'Velocity only':      ['velocity_log'],
        'Location only':      LOCATION,
        'Our six features':   SIX,
        'Six + location':     SIX + [c for c in LOCATION if c not in SIX],
        'All native columns': NATIVE_FEATURES,
    }

def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=N_TREES, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

def train_and_score(train_sub, val, test, ds, GROUPS):
    y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
    prevalence = y_test.mean()
    test_days = test['day'].values
    rows = []
    for g, cols in GROUPS.items():
        seed_runs = run_seeds(f'{ds} {g}', make_rf(len(cols)),
                              train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                              checkpoint_dir=checkpoint_dir(ds))
        pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
        boot = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
        s = summarise(y_test, pairs, boot)
        rows.append({'Dataset': ds, 'Feature group': g, 'Features': len(cols),
                     'Seeds': len(SEEDS), 'Test rows': len(y_test), 'Test fraud': int(y_test.sum()),
                     'Prevalence': prevalence,
                     'Val PR-AUC': np.mean([r['val_pr_auc'] for r in seed_runs]),
                     'PR-AUC': s['PR-AUC'], 'PR-AUC CI low': s['PR-AUC CI low'], 'PR-AUC CI high': s['PR-AUC CI high'],
                     'Lift': s['PR-AUC'] / prevalence,
                     'Lift CI low': s['PR-AUC CI low'] / prevalence, 'Lift CI high': s['PR-AUC CI high'] / prevalence,
                     'F2': s['F2'], 'F2 CI low': s['F2 CI low'], 'F2 CI high': s['F2 CI high'],
                     'MCC': s['MCC'], 'MCC CI low': s['MCC CI low'], 'MCC CI high': s['MCC CI high'],
                     'Thresholds': s['Thresholds']})
        print(f'{g}: done')
    return pd.DataFrame(rows)

def results_table(result):
    return pd.DataFrame({
        'PR-AUC [95% CI]': [f"{r['PR-AUC']:.4f} [{r['PR-AUC CI low']:.4f}, {r['PR-AUC CI high']:.4f}]" for _, r in result.iterrows()],
        'Lift [95% CI]':   [f"{r['Lift']:.1f} [{r['Lift CI low']:.1f}, {r['Lift CI high']:.1f}]" for _, r in result.iterrows()],
        'F2 [95% CI]':     [f"{r['F2']:.4f} [{r['F2 CI low']:.4f}, {r['F2 CI high']:.4f}]" for _, r in result.iterrows()],
        'MCC [95% CI]':    [f"{r['MCC']:.4f} [{r['MCC CI low']:.4f}, {r['MCC CI high']:.4f}]" for _, r in result.iterrows()],
    }, index=result['Feature group'])

In [10]:
# 2.6 TabFormer: load, build the features, split and encode
ds = 'tabformer'
d, NATIVE_COLS, TEXT_COLS = LOADERS[ds]()
d, LOCATION = build_features(d, ds)
train_sub, val, test = split_by_time(d, ds)
del d; gc.collect()
NATIVE_FEATURES, native_info = encode(train_sub, val, test, ds, NATIVE_COLS, TEXT_COLS)
GROUPS = feature_groups(LOCATION, NATIVE_FEATURES)
y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
PREVALENCE = y_test.mean()
test_days = test['day'].values
display(native_info)

train_sub 2015-01-01 00:01:00 to 2018-04-21 09:28:00  rows=1,168,504  fraud= 1,511  rate=0.1293%
val       2018-04-21 09:29:00 to 2019-02-16 23:50:00  rows=  293,511  fraud=   362  rate=0.1233%
test      2019-02-17 00:00:00 to 2020-02-28 23:49:00  rows=  379,534  fraud=   309  rate=0.0814%
all-native run uses 18 columns


,column,used as
0,Amount,numeric (parsed text)
1,Use Chip,fraud rate of each value
2,Merchant City,"left out (more than 1,000 values)"
3,Merchant State,fraud rate of each value
4,MCC,fraud rate of each value
5,Errors?,fraud rate of each value
6,Current Age,numeric
7,Retirement Age,numeric
8,Birth Year,numeric
9,Gender,fraud rate of each value


## 3. Each native column on its own
We score every native column on its own, without a model: the test transactions are ranked by the column's value (numeric columns as they are, text columns by the fraud rate of each value on the training part). We flip a numeric column if lower values mean more fraud on the training part. A single column with a very high score would be a sign of leakage.

In [11]:
# 3.1 PR-AUC, lift and ROC-AUC of every native column on its own
from sklearn.metrics import average_precision_score, roc_auc_score
rows = []
for name in NATIVE_FEATURES:
    x_fit, x_test = train_sub[name], test[name]
    if x_fit.nunique() > 1 and roc_auc_score(y_tr, x_fit) < 0.5:
        x_test = -x_test
    pr = average_precision_score(y_test, x_test)
    rows.append({'column': name.replace('raw__', ''), 'PR-AUC': pr, 'lift': pr / PREVALENCE,
                 'ROC-AUC': roc_auc_score(y_test, x_test)})
single = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)
single.round(4)

,column,PR-AUC,lift,ROC-AUC
0,Merchant State,0.7843,963.2843,0.9999
1,MCC,0.0108,13.2946,0.7952
2,City,0.0030,3.6962,0.6266
3,Amount,0.0019,2.3192,0.5559
4,Longitude,0.0014,1.6639,0.4932
5,State,0.0014,1.6630,0.5592
6,Current Age,0.0012,1.4597,0.5878
7,Birth Year,0.0012,1.4529,0.5877
8,Yearly Income - Person,0.0012,1.4379,0.4641
9,Latitude,0.0011,1.3560,0.5093


## 4. The `Errors?` column
For each error value we count the transactions and the fraud rate on the training part and on the test part. An error value that appears almost only with fraud, in both periods, would be leakage.

In [12]:
# 4.1 Fraud rate of each Errors? value
def error_table(part):
    e = part['Errors?'].fillna('(no error)').astype(str)
    return part.groupby(e)['_y'].agg(transactions='size', fraud='sum', fraud_rate='mean')

errors = error_table(train_sub).add_prefix('train ').join(error_table(test).add_prefix('test '), how='outer')
errors = errors.sort_values('train transactions', ascending=False)
print(f"transactions with an error: train {train_sub['Errors?'].notna().mean():.2%}, test {test['Errors?'].notna().mean():.2%}")
print(f"fraud with an error: train {train_sub.loc[y_tr == 1, 'Errors?'].notna().mean():.2%}, "
      f"test {test.loc[y_test == 1, 'Errors?'].notna().mean():.2%}")
errors.round(4)

transactions with an error: train 1.60%, test 1.61%
fraud with an error: train 4.83%, test 2.27%


,train transactions,train fraud,train fraud_rate,test transactions,test fraud,test fraud_rate
Errors?,,,,,,
(no error),1149785,1438,0.0013,373427.0,302.0,0.0008
Insufficient Balance,11499,24,0.0021,3786.0,2.0,0.0005
Bad PIN,2860,9,0.0031,911.0,5.0,0.0055
Technical Glitch,2358,3,0.0013,781.0,0.0,0.0000
Bad Card Number,711,9,0.0127,211.0,0.0,0.0000
Bad Expiration,568,11,0.0194,174.0,0.0,0.0000
Bad CVV,562,17,0.0302,184.0,0.0,0.0000
Bad Zipcode,84,0,0.0000,34.0,0.0,0.0000
"Bad PIN,Insufficient Balance",24,0,0.0000,9.0,0.0,0.0000


## 5. Merchants and users seen in training
`Merchant Name` is TabFormer's merchant identifier. The native run never uses it (nor `Zip`), and it leaves out `Merchant City` because it has more than 1,000 values, so the model cannot memorise individual merchants. We still count how often test fraud happens at a merchant that already had fraud in the training period, and how many test users also appear in training.

In [13]:
# 5.1 Merchant names and users seen in training
fraud_merchants_train = set(train_sub.loc[y_tr == 1, 'Merchant Name'].astype(str))
merchants_train = set(train_sub['Merchant Name'].astype(str))
test_fraud_merchant = test.loc[y_test == 1, 'Merchant Name'].astype(str)
users_train = set(train_sub['User']) | set(val['User'])
pd.Series({
    'distinct merchants in training': len(merchants_train),
    'test fraud at a merchant seen in training %': test_fraud_merchant.isin(merchants_train).mean() * 100,
    'test fraud at a merchant that had fraud in training %': test_fraud_merchant.isin(fraud_merchants_train).mean() * 100,
    'test normal transactions at a merchant that had fraud in training %':
        test.loc[y_test == 0, 'Merchant Name'].astype(str).isin(fraud_merchants_train).mean() * 100,
    'users in test': test['User'].nunique(),
    'test users also in training %': test['User'].isin(users_train).groupby(test['User']).first().mean() * 100,
    'test fraud from users also in training %': test.loc[y_test == 1, 'User'].isin(users_train).mean() * 100,
}).round(2).to_frame('value')

,value
distinct merchants in training,16959.00
test fraud at a merchant seen in training %,100.00
test fraud at a merchant that had fraud in training %,96.76
test normal transactions at a merchant that had fraud in training %,42.36
users in test,393.00
test users also in training %,78.88
test fraud from users also in training %,100.00


## 6. Retraining without `Errors?` and without the merchant fields
We train the native-column model again with five seeds, once without `Errors?` and once without `Merchant State`. The full native run is loaded from the seeds `cross_dataset_protocol.ipynb` saved.

In [14]:
# 6.1 The native runs
def drop(cols, names):
    return [c for c in cols if c.replace('raw__', '') not in names]

RUNS = {
    'All native columns':      GROUPS['All native columns'],
    'Without Errors?':         drop(GROUPS['All native columns'], {'Errors?'}),
    'Without merchant fields': drop(GROUPS['All native columns'], {'Merchant State', 'Merchant City'}),
}
for name, cols in RUNS.items():
    print(f'{name}: {len(cols)} columns')

pairs = {}
for name, cols in RUNS.items():
    run_name = f'{ds} {name}'   # 'tabformer All native columns' is the saved cross_dataset_protocol run
    seed_runs = run_seeds(run_name, make_rf(len(cols)), train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                          checkpoint_dir=checkpoint_dir(ds))
    pairs[name] = [(r['test_proba'], r['threshold']) for r in seed_runs]

All native columns: 18 columns
Without Errors?: 17 columns
Without merchant fields: 17 columns
tabformer All native columns seed 42: loaded from checkpoint
tabformer All native columns seed 43: loaded from checkpoint
tabformer All native columns seed 44: loaded from checkpoint
tabformer All native columns seed 45: loaded from checkpoint
tabformer All native columns seed 46: loaded from checkpoint
tabformer Without Errors? seed 42: loaded from checkpoint
tabformer Without Errors? seed 43: loaded from checkpoint
tabformer Without Errors? seed 44: loaded from checkpoint
tabformer Without Errors? seed 45: loaded from checkpoint
tabformer Without Errors? seed 46: loaded from checkpoint
tabformer Without merchant fields seed 42: loaded from checkpoint
tabformer Without merchant fields seed 43: loaded from checkpoint
tabformer Without merchant fields seed 44: loaded from checkpoint
tabformer Without merchant fields seed 45: loaded from checkpoint
tabformer Without merchant fields seed 46: loa

## 7. Retraining with the test users left out
We split the users into five groups. For each group we train a model on the training and validation transactions of the other four groups only, choose its threshold on their validation transactions, and use it to score the test transactions of the left-out group. Every test transaction is then scored by a model that never saw its user. We use one seed per group (42 to 46), so the total number of models is the same as in a normal five-seed run. Each model sees about 80% of the users, so some drop is expected even without leakage. `category_risk` and the native-column encodings stay as fitted on the full training part, so the left-out users' training transactions still count towards the fraud rate of each category; the models themselves never see those users' transactions.

In [15]:
# 7.1 Five user groups, one model per group
rng = np.random.default_rng(FOLD_SEED)
users = np.array(sorted(test['User'].unique()))
fold_of_user = pd.Series(rng.permutation(len(users)) % USER_FOLDS, index=users)

def unseen_users(group, cols):
    proba = np.zeros(len(test))
    threshold = np.zeros(len(test))
    for k in range(USER_FOLDS):
        left_out = set(fold_of_user.index[fold_of_user == k])
        tr = ~train_sub['User'].isin(left_out)
        va = ~val['User'].isin(left_out)
        te = test['User'].isin(left_out).values
        # category_risk and the native encodings stay as fitted on the full training part
        r = run_seeds(f'{ds} {group}, unseen users fold {k}', make_rf(len(cols)),
                      train_sub.loc[tr, cols], y_tr[tr], val.loc[va, cols], y_val[va], test.loc[te, cols],
                      [SEEDS[k % len(SEEDS)]], checkpoint_dir=checkpoint_dir(ds))[0]
        proba[te] = r['test_proba']
        threshold[te] = r['threshold']
    return [(proba, threshold)]

for group in ['All native columns', 'Six + location']:
    pairs[f'Unseen users, {group.lower()}'] = unseen_users(group, GROUPS[group])

seen_six = run_seeds(f'{ds} Six + location', make_rf(len(GROUPS['Six + location'])),
                     train_sub[GROUPS['Six + location']], y_tr, val[GROUPS['Six + location']], y_val,
                     test[GROUPS['Six + location']], SEEDS, checkpoint_dir=checkpoint_dir(ds))
pairs['Six + location'] = [(r['test_proba'], r['threshold']) for r in seen_six]

tabformer All native columns, unseen users fold 0 seed 42: loaded from checkpoint
tabformer All native columns, unseen users fold 1 seed 43: loaded from checkpoint
tabformer All native columns, unseen users fold 2 seed 44: loaded from checkpoint
tabformer All native columns, unseen users fold 3 seed 45: loaded from checkpoint
tabformer All native columns, unseen users fold 4 seed 46: loaded from checkpoint
tabformer Six + location, unseen users fold 0 seed 42: loaded from checkpoint
tabformer Six + location, unseen users fold 1 seed 43: loaded from checkpoint
tabformer Six + location, unseen users fold 2 seed 44: loaded from checkpoint
tabformer Six + location, unseen users fold 3 seed 45: loaded from checkpoint
tabformer Six + location, unseen users fold 4 seed 46: loaded from checkpoint
tabformer Six + location seed 42: loaded from checkpoint
tabformer Six + location seed 43: loaded from checkpoint
tabformer Six + location seed 44: loaded from checkpoint
tabformer Six + location seed

## 8. Results
Every run is bootstrapped on the same 500 resampled test days. Each run is compared with the model it changes: the native runs with the full native run, and the unseen-user runs with the same feature group trained on all users. A drop whose interval is above zero is a real loss.

In [16]:
# 8.1 Bootstrap and the differences
boots, rows = {}, []
for name, p in pairs.items():
    boots[name] = bootstrap_model(y_test, p, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    per_seed = pd.DataFrame([point_metrics(y_test, proba, t) for proba, t in p])
    rows.append({'Run': name, 'Models': len(p),
                 'PR-AUC': per_seed['PR-AUC'].mean(),
                 'PR-AUC CI low': np.percentile(boots[name]['PR-AUC'], 2.5),
                 'PR-AUC CI high': np.percentile(boots[name]['PR-AUC'], 97.5),
                 'Lift': per_seed['PR-AUC'].mean() / PREVALENCE,
                 'F2': per_seed['F2'].mean(), 'MCC': per_seed['MCC'].mean()})
    print(f'{name}: done')
result = pd.DataFrame(rows).set_index('Run')

BASELINE = {'Without Errors?': 'All native columns', 'Without merchant fields': 'All native columns',
            'Unseen users, all native columns': 'All native columns',
            'Unseen users, six + location': 'Six + location'}
for name, base in BASELINE.items():
    dd = paired_difference(boots[base], boots[name], 'PR-AUC')
    result.loc[name, ['Compared with', 'Drop', 'Drop CI low', 'Drop CI high']] = [base, dd['mean diff'], dd['CI low'], dd['CI high']]

out = f"{RESULTS_DIR}/point5_tabformer_leakage{'_quick' if QUICK else ''}.csv"
result.to_csv(out)
os.makedirs('results', exist_ok=True)
result.to_csv(f"results/point5_tabformer_leakage{'_quick' if QUICK else ''}.csv")
print('saved ->', out)

order = ['All native columns', 'Without Errors?', 'Without merchant fields', 'Unseen users, all native columns',
         'Six + location', 'Unseen users, six + location']
pd.DataFrame({
    'PR-AUC [95% CI]': [f"{result.loc[n, 'PR-AUC']:.4f} [{result.loc[n, 'PR-AUC CI low']:.4f}, {result.loc[n, 'PR-AUC CI high']:.4f}]" for n in order],
    'Lift': [round(result.loc[n, 'Lift'], 1) for n in order],
    'F2': [round(result.loc[n, 'F2'], 4) for n in order],
    'MCC': [round(result.loc[n, 'MCC'], 4) for n in order],
    'Drop [95% CI]': ['' if n not in BASELINE else
                      f"{result.loc[n, 'Drop']:.4f} [{result.loc[n, 'Drop CI low']:.4f}, {result.loc[n, 'Drop CI high']:.4f}] vs {BASELINE[n].lower()}"
                      for n in order],
}, index=order)

All native columns: done
Without Errors?: done
Without merchant fields: done
Unseen users, all native columns: done
Unseen users, six + location: done
Six + location: done
saved -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results/point5_tabformer_leakage.csv


,PR-AUC [95% CI],Lift,F2,MCC,Drop [95% CI]
All native columns,"0.8867 [0.8505, 0.9232]",1089.0,0.8308,0.7737,
Without Errors?,"0.8814 [0.8439, 0.9194]",1082.7,0.8244,0.7502,"0.0053 [0.0024, 0.0089] vs all native columns"
Without merchant fields,"0.0032 [0.0023, 0.0048]",3.9,0.0278,0.0209,"0.8845 [0.8476, 0.9194] vs all native columns"
"Unseen users, all native columns","0.8584 [0.8010, 0.9059]",1054.3,0.8162,0.7177,"0.0294 [0.0072, 0.0578] vs all native columns"
Six + location,"0.3456 [0.2831, 0.4070]",424.5,0.4125,0.3350,
"Unseen users, six + location","0.3191 [0.2638, 0.3746]",391.9,0.3874,0.3113,"0.0274 [0.0074, 0.0491] vs six + location"
